In [22]:
# !pip install google-genai

In [23]:
# Importing necessary libraries
import numpy as np
import uuid
import umap
import shap
import pandas as pd
import faker
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import classification_report, accuracy_score
import datetime

from arize.pandas.logger import Client
from arize.utils.types import ModelTypes, Environments, Schema, Metrics, Embedding, EmbeddingColumnNames
import numpy as np
import pandas as pd
from faker import Faker


RANDOM_STATE = 123
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

def get_uuid():
    return uuid.uuid4().hex 
    
def random_dates(start, end, n=10):

    start_u = start.value//10**9
    end_u = end.value//10**9
    
    return pd.to_datetime(np.random.randint(start_u, end_u, n), unit='s')

fake = Faker()

In [24]:
# Load the breast cancer dataset
data = load_breast_cancer()

# Converting the dataset into a pandas DataFrame for easier handling
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # Add the target column (0 for malignant, 1 for benign)
df['target'] = df['target'].map({0: 'benign', 1: 'malignant'}) 

In [25]:
# Create a primary key for the dataset
df['prediction_id'] = [get_uuid() for _ in range(df.shape[0])]

start = pd.to_datetime('2025-03-01')
end = pd.to_datetime('2025-03-15')

df['event_timestamp'] = random_dates(start, end, df.shape[0])

df['state'] = [fake.state() for _ in range(df.shape[0])]

In [26]:
# Splitting the dataset into training and testing sets (80% training, 20% testing)
X = df.drop('target', axis=1)  # Features
y = df['target']               # Target

features = X.drop(['prediction_id', 'event_timestamp', 'state'], axis=1).columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

In [27]:
# Initialize the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# Fit the model to the training data
rf_classifier.fit(X_train[features], y_train)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test[features])

# Evaluate the model's performance using accuracy and classification report
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9912

Classification Report:
              precision    recall  f1-score   support

      benign       1.00      0.98      0.99        41
   malignant       0.99      1.00      0.99        73

    accuracy                           0.99       114
   macro avg       0.99      0.99      0.99       114
weighted avg       0.99      0.99      0.99       114



In [28]:
# Create the SHAP explainer for the trained model
explainer = shap.TreeExplainer(rf_classifier)

# Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test[features])

shap_values_positive_class = shap_values[:,:,1]

# 6. Convert the SHAP values to a DataFrame
shap_df = pd.DataFrame(shap_values_positive_class, columns=features)

# Because we are doing a bulk upload we need to rename the columns by added _shap at the end
shap_dataframe = pd.DataFrame(
        shap_values_positive_class, columns=[f"{fn}_shap" for fn in features], index=X_test.index
)

# Create a mapping
mapping = {fn: f"{fn}_shap" for fn in X[features].columns}

shap_dataframe.head()

,mean radius_shap,mean texture_shap,mean perimeter_shap,mean area_shap,mean smoothness_shap,mean compactness_shap,mean concavity_shap,mean concave points_shap,mean symmetry_shap,mean fractal dimension_shap,...,worst radius_shap,worst texture_shap,worst perimeter_shap,worst area_shap,worst smoothness_shap,worst compactness_shap,worst concavity_shap,worst concave points_shap,worst symmetry_shap,worst fractal dimension_shap
333,0.006227,0.012070,0.010931,0.011793,0.002703,0.002851,0.023857,0.051082,0.000684,0.000399,...,0.046181,0.008758,0.039769,0.054636,0.005563,0.007451,0.019355,0.041595,0.001851,0.000665
273,0.008022,0.007974,0.013161,0.017900,0.001117,0.002766,0.023050,0.047972,-0.000969,0.000877,...,0.047510,0.013139,0.043722,0.056637,-0.002086,0.006957,0.020438,0.042495,0.001245,0.001043
201,-0.023076,-0.002633,-0.033997,-0.033522,0.001392,-0.001851,-0.030186,-0.087250,-0.000824,-0.002244,...,-0.095546,-0.000588,-0.065963,-0.108055,0.000095,-0.003081,-0.020623,-0.088405,-0.000041,-0.000516
178,0.009886,-0.020028,0.009680,0.011953,0.009469,0.003560,0.024270,0.060509,0.001172,-0.007354,...,0.044889,0.000599,0.034879,0.049116,0.014614,0.007606,0.018536,0.041735,0.006710,-0.001883
85,-0.023417,-0.000158,-0.031711,-0.032024,-0.000701,0.002015,-0.035971,-0.085456,-0.001582,-0.000027,...,-0.094815,-0.006637,-0.067245,-0.110220,-0.001136,-0.001609,-0.015189,-0.073418,-0.002053,0.000660


In [29]:
embedder = umap.UMAP(random_state=RANDOM_STATE, n_components=3)

X_train_embeddings = embedder.fit_transform(X_train[features], y_train.map({'benign': 0, 'malignant': 1}))

In [35]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "/Users/carterdaly/Documents/GitHub/PredictionMarketArb/data/model_cache/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

# Sort SHAP values by absolute importance (descending) and take top 5
shap_pairs = list(zip(features, explainer.shap_values(pd.DataFrame([X_train[features].iloc[0].to_dict()]))[:,:,1][0]))
top_shap = sorted(shap_pairs, key=lambda x: abs(x[1]), reverse=True)[:5]
shap_summary = "\n".join(
    f"  - {feat}: {val:+.4f} ({'increases' if val > 0 else 'decreases'} malignancy probability)"
    for feat, val in top_shap
)

prediction = rf_classifier.predict(pd.DataFrame([X_train[features].iloc[0].to_dict()]))[0]
actual_label = y_train.iloc[0]

system_prompt = (
    "You are a clinical decision-support assistant helping oncologists interpret "
    "machine learning predictions on breast cancer biopsies. "
    "You explain model outputs in clear, precise language grounded in the provided data. "
    "Never speculate beyond what the data supports. Never give a diagnosis."
)

user_prompt = f"""A Random Forest classifier analyzed a breast cancer biopsy and produced the following result.

PREDICTION: {prediction}
ACTUAL LABEL: {actual_label}

TOP 5 MOST INFLUENTIAL FEATURES (SHAP values):
{shap_summary}

Think step by step:
1. State whether the prediction matches the actual label and what that means.
2. Explain what each of the top SHAP features is measuring in plain language.
3. Describe how each feature pushed the model toward or away from a malignant prediction.
4. Summarize in 1-2 sentences what a clinician should take away from this result.

Be concise and factual."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=400,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.1,
    do_sample=True,
)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5307.00it/s]


The prediction for benign was made based on the analysis of the breast cancer biopsy, which showed a benign outcome. The model's output indicates that the most influential factors were:

1. **Worst Concave Points**: These points decrease the likelihood of malignancy, suggesting they might be related to genetic predispositions or other factors influencing the outcome.
   
2. **Mean Concave Points**: These also decrease the likelihood of malignancy, indicating similar patterns but with slightly different weights.

3. **Worst Texture**: This factor decreases the likelihood of malignancy, possibly due to skin conditions affecting tumor growth.

4. **Mean Concavity**: Similar to the previous point, it decreases the likelihood of malignancy, though with slight differences in impact.

These features collectively suggest that while there are no direct signs of malignancy, certain genetic or environmental factors may influence the outcome more than others. The model leans towards predicting ben

In [33]:
!pip install transformers torch tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 994.0/994.0 kB 15.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
X_train.iloc[0].to_dict()

{'mean radius': 14.22,
 'mean texture': 23.12,
 'mean perimeter': 94.37,
 'mean area': 609.9,
 'mean smoothness': 0.1075,
 'mean compactness': 0.2413,
 'mean concavity': 0.1981,
 'mean concave points': 0.06618,
 'mean symmetry': 0.2384,
 'mean fractal dimension': 0.07542,
 'radius error': 0.286,
 'texture error': 2.11,
 'perimeter error': 2.112,
 'area error': 31.72,
 'smoothness error': 0.00797,
 'compactness error': 0.1354,
 'concavity error': 0.1166,
 'concave points error': 0.01666,
 'symmetry error': 0.05113,
 'fractal dimension error': 0.01172,
 'worst radius': 15.74,
 'worst texture': 37.18,
 'worst perimeter': 106.4,
 'worst area': 762.4,
 'worst smoothness': 0.1533,
 'worst compactness': 0.9327,
 'worst concavity': 0.8488,
 'worst concave points': 0.1772,
 'worst symmetry': 0.5166,
 'worst fractal dimension': 0.1446,
 'prediction_id': 'c6ea3706b66449d58d1ba7049af96c60',
 'event_timestamp': Timestamp('2025-03-08 14:15:17'),
 'state': 'Minnesota'}

In [ ]:
y_train[0]

'benign'

In [ ]:
from scipy.spatial.distance import cdist

def closest_node(node, nodes):
    return nodes[cdist([node], nodes).argmin()]

In [ ]:
target = closest_node(X_train_embeddings[0], X_train_embeddings[1:])
target

array([-5.5748  ,  8.880056,  5.838913], dtype=float32)

In [ ]:
np.where(X_train_embeddings == target)

(array([409, 409, 409]), array([0, 1, 2]))

In [ ]:
prompt_values = {
    'features': X_train[features].iloc[0].to_dict()
    , 'label': y_train[0]
    , 'prediction': rf_classifier.predict(pd.DataFrame([X_train[features].iloc[0].to_dict()]))[0]
    , 'shap_values': zip(features, explainer.shap_values(pd.DataFrame([X_train[features].iloc[0].to_dict()]))[:,:,1])
    , 'embedding': []
}

ChatCompletion(id='chatcmpl-BHZa123Kn59dEXa4c0hFFr83n8Zn9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Silent circuits hum,  \nThoughts woven from code and light,  \nDreams in binary.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1743528045, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_b376dfbbd5', usage=CompletionUsage(completion_tokens=20, prompt_tokens=13, total_tokens=33, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [ ]:
rf_classifier.predict(pd.DataFrame([X_train[features].iloc[0].to_dict()]))[0]

'benign'

In [ ]:
X_train_embeddings.index(target)

AttributeError: 'numpy.ndarray' object has no attribute 'index'

(array([409, 409, 409]), array([0, 1, 2]))

In [ ]:
X_train_embeddings[409]

array([-5.5748  ,  8.880056,  5.838913], dtype=float32)